# Notebook 6.1: Understanding Simple Linear Regression

**Companion to Chapter 6: Simple Linear Regression**  
*Machine Learning with Python: Principles and Practical Techniques*

> **Estimated time:** 40–50 minutes  
> **Level:** Beginner  
> **Environment:** Google Colab or Jupyter Notebook

---

## Related chapter ideas

This notebook introduces the need for simple linear regression, independent and dependent variables, the straight-line hypothesis, slope, intercept, prediction error, and the cost function.

## Learning objectives

By the end of this notebook, you will be able to:

1. identify independent and dependent variables;
2. use a scatter plot to examine a possible linear relationship;
3. express a simple linear regression hypothesis;
4. interpret the slope and intercept in context;
5. calculate predictions and residuals;
6. compute mean squared error and the chapter cost function; and
7. explain why regression minimizes squared errors rather than merely drawing a visually appealing line.


## What will you build?

You will investigate whether a researcher's experience can help explain their annual stipend. Starting from a small dataset, you will construct and compare candidate straight lines and then calculate the best-fitting line directly.

The learning journey is:

**Data → Scatter plot → Hypothesis → Predictions → Residuals → Cost → Best-fitting line**

> **Responsible practice:** The dataset is synthetic and intended only for learning. Compensation decisions should never be based on experience alone; discipline, responsibilities, location, funding, equity, and other contextual factors also matter.


## 1. Import the libraries


In [ ]:
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)
print("Pandas version:", pd.__version__)


## 2. Load the research-experience dataset


In [ ]:
stipend_csv = """experience_years,annual_stipend_thousands
0.5,34
1.0,37
1.5,39
2.0,43
2.5,45
3.0,49
3.5,50
4.0,54
4.5,57
5.0,60
5.5,61
6.0,66
6.5,68
7.0,70
7.5,74
8.0,77
8.5,79
9.0,82
9.5,85
10.0,88
"""

stipends = pd.read_csv(StringIO(stipend_csv))
print("Dataset shape:", stipends.shape)
display(stipends.head())


The stipend is measured in thousands of US dollars. For example, `34` represents an annual stipend of **$34,000**.


In [ ]:
stipends.info()
display(stipends.describe().T)


## 3. Identify the variables


In [ ]:
X = stipends["experience_years"].to_numpy()
y = stipends["annual_stipend_thousands"].to_numpy()

print("Independent variable X: research experience in years")
print("Dependent variable y: annual stipend in thousands of dollars")
print("Number of observations:", len(X))


- The **independent variable** $x$ is the predictor or explanatory variable.
- The **dependent variable** $y$ is the outcome or response variable.

Simple linear regression uses exactly one input feature. It asks whether a straight line can summarize the average relationship between $x$ and $y$.


## 4. Visualize the relationship


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X, y, s=75, color="#4C78A8", edgecolor="black", alpha=0.85)
ax.set_title("Research Experience and Annual Stipend")
ax.set_xlabel("Research experience (years)")
ax.set_ylabel("Annual stipend ($ thousands)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


### Pause and observe

1. Does the relationship appear positive, negative, or absent?
2. Would a straight line provide a reasonable summary?
3. Do all points lie exactly on one line?
4. Can this plot establish that experience *causes* a higher stipend?

The points show a strong positive pattern, but association alone does not establish causation.


## 5. Express the regression hypothesis


The straight-line hypothesis is:

$$\hat{y} = \theta_0 + \theta_1x$$

where:

- $\hat{y}$ is the predicted value of the dependent variable;
- $x$ is the independent variable;
- $\theta_0$ is the intercept; and
- $\theta_1$ is the slope or coefficient.

The parameters $\theta_0$ and $\theta_1$ determine the position and direction of the line.


In [ ]:
def predict_line(x_values, intercept, slope):
    # Return predictions from a straight-line hypothesis.
    return intercept + slope * np.asarray(x_values)


candidate_intercept = 30
candidate_slope = 5.5
candidate_predictions = predict_line(X, candidate_intercept, candidate_slope)

print("Candidate hypothesis: stipend = 30 + 5.5 × experience")
print("Prediction for 4 years:", predict_line(4, candidate_intercept, candidate_slope), "thousand dollars")


## 6. Interpret slope and intercept


For the candidate hypothesis

$$\widehat{stipend} = 30 + 5.5( experience),$$

- the **intercept**, 30, predicts a stipend of $30,000 at zero years of experience;
- the **slope**, 5.5, predicts an average increase of $5,500 for each additional year of experience.

The intercept is mathematically necessary, but it is meaningful only when $x=0$ is plausible and sufficiently close to the observed data. Extrapolating far outside the observed experience range is risky.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X, y, s=70, color="#4C78A8", edgecolor="black", label="Observed data")
ax.plot(X, candidate_predictions, color="#E45756", linewidth=2.5,
        label="Candidate line: 30 + 5.5x")
ax.set_title("A Candidate Regression Line")
ax.set_xlabel("Research experience (years)")
ax.set_ylabel("Annual stipend ($ thousands)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 7. Calculate residuals


A **residual** is the vertical difference between an observed and predicted value:

$$e_i = y_i - \hat{y}_i$$

- A positive residual means the observed value is above the line.
- A negative residual means the observed value is below the line.
- A residual of zero means the line predicts that observation exactly.


In [ ]:
residuals = y - candidate_predictions

prediction_table = pd.DataFrame({
    "experience_years": X,
    "actual_stipend": y,
    "predicted_stipend": candidate_predictions,
    "residual": residuals,
    "squared_error": residuals ** 2,
})

display(prediction_table.head(10))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X, y, s=70, color="#4C78A8", edgecolor="black", zorder=3)
ax.plot(X, candidate_predictions, color="#E45756", linewidth=2.2)

for x_value, actual, predicted in zip(X, y, candidate_predictions):
    ax.plot([x_value, x_value], [predicted, actual], color="gray", alpha=0.65)

ax.set_title("Residuals as Vertical Distances")
ax.set_xlabel("Research experience (years)")
ax.set_ylabel("Annual stipend ($ thousands)")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 8. Define the cost function


Squaring residuals prevents positive and negative errors from canceling and penalizes larger errors more strongly.

The **mean squared error** is:

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)^2$$

The chapter cost function is commonly written as:

$$J(\theta_0,\theta_1)=\frac{1}{2n}\sum_{i=1}^{n}(\hat{y}_i-y_i)^2$$

Therefore, $J = MSE/2$. The factor $1/2$ simplifies the derivative used in gradient descent but does not change which line minimizes the function.


In [ ]:
def mean_squared_error(y_true, y_pred):
    errors = np.asarray(y_pred) - np.asarray(y_true)
    return np.mean(errors ** 2)


def chapter_cost(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) / 2


candidate_mse = mean_squared_error(y, candidate_predictions)
candidate_cost = chapter_cost(y, candidate_predictions)

print("Candidate-line MSE:", round(candidate_mse, 3))
print("Candidate-line cost J:", round(candidate_cost, 3))


## 9. Compare several candidate lines


In [ ]:
candidates = pd.DataFrame([
    {"line": "A", "intercept": 28.0, "slope": 4.0},
    {"line": "B", "intercept": 30.0, "slope": 5.5},
    {"line": "C", "intercept": 32.0, "slope": 6.0},
])

candidates["cost_J"] = candidates.apply(
    lambda row: chapter_cost(y, predict_line(X, row["intercept"], row["slope"])),
    axis=1,
)

display(candidates)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X, y, s=70, color="black", label="Observed data", zorder=3)

colors = ["#F58518", "#4C78A8", "#54A24B"]
for (_, row), color in zip(candidates.iterrows(), colors):
    predictions = predict_line(X, row["intercept"], row["slope"])
    ax.plot(X, predictions, color=color, linewidth=2,
            label=f"Line {row['line']} (J={row['cost_J']:.2f})")

ax.set_title("Candidate Lines and Their Costs")
ax.set_xlabel("Research experience (years)")
ax.set_ylabel("Annual stipend ($ thousands)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


The line with the lowest cost is the best among these three candidates. However, comparing a few guesses does not guarantee that we have found the overall minimum.


## 10. See the shape of cost for one parameter


In [ ]:
# Hold the intercept fixed and vary only the slope.
fixed_intercept = 30
slope_values = np.linspace(2, 9, 200)
slope_costs = [
    chapter_cost(y, predict_line(X, fixed_intercept, slope))
    for slope in slope_values
]

best_grid_position = int(np.argmin(slope_costs))
best_grid_slope = slope_values[best_grid_position]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(slope_values, slope_costs, color="#4C78A8", linewidth=2.5)
ax.scatter(best_grid_slope, slope_costs[best_grid_position],
           color="#E45756", s=90, edgecolor="black", zorder=3)
ax.set_title("Cost as the Slope Changes (Intercept Fixed at 30)")
ax.set_xlabel("Slope θ₁")
ax.set_ylabel("Cost J")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("Best slope on this grid:", round(best_grid_slope, 3))


The curve is convex: it has one global minimum. Notebook 6.2 will show how gradient descent moves toward this minimum without checking every possible parameter value.


## 11. Calculate the least-squares line directly


For one feature, the ordinary least-squares parameters can be calculated directly:

$$\theta_1 = \frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sum (x_i-\bar{x})^2}$$

$$\theta_0 = \bar{y}-\theta_1\bar{x}$$

These values minimize the sum of squared residuals.


In [ ]:
x_mean = X.mean()
y_mean = y.mean()

best_slope = np.sum((X - x_mean) * (y - y_mean)) / np.sum((X - x_mean) ** 2)
best_intercept = y_mean - best_slope * x_mean
best_predictions = predict_line(X, best_intercept, best_slope)

print("Least-squares intercept θ₀:", round(best_intercept, 3))
print("Least-squares slope θ₁:", round(best_slope, 3))
print("Least-squares cost J:", round(chapter_cost(y, best_predictions), 3))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X, y, s=75, color="#4C78A8", edgecolor="black", label="Observed data")
ax.plot(X, best_predictions, color="#E45756", linewidth=2.5,
        label=f"Best line: {best_intercept:.2f} + {best_slope:.2f}x")
ax.set_title("Least-Squares Regression Line")
ax.set_xlabel("Research experience (years)")
ax.set_ylabel("Annual stipend ($ thousands)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 12. Interpret the fitted parameters


In [ ]:
example_experience = 6
example_prediction = predict_line(example_experience, best_intercept, best_slope)

print(f"For each additional year of research experience, the model predicts")
print(f"an average stipend increase of about ${best_slope * 1000:,.0f}.")
print(f"At {example_experience} years, the predicted annual stipend is")
print(f"approximately ${example_prediction * 1000:,.0f}.")


The slope describes the fitted association in this dataset. It does **not** prove that adding one year of experience causes the stated stipend increase.


## 13. Interpolation versus extrapolation


In [ ]:
prediction_examples = pd.DataFrame({
    "experience_years": [4, 12, 25]
})
prediction_examples["predicted_stipend_thousands"] = predict_line(
    prediction_examples["experience_years"], best_intercept, best_slope
)
prediction_examples["type"] = ["Interpolation", "Extrapolation", "Extreme extrapolation"]

display(prediction_examples)


- **Interpolation** predicts within the observed range of 0.5–10 years.
- **Extrapolation** predicts beyond that range and assumes the same linear pattern continues.

A mathematically valid prediction can still be practically unreasonable. The 25-year prediction is especially uncertain because the notebook contains no data near that experience level.


## 14. Examine residuals from the best-fitting line


In [ ]:
best_residuals = y - best_predictions

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(best_predictions, best_residuals, s=75,
           color="#4C78A8", edgecolor="black")
ax.axhline(0, color="#E45756", linestyle="--", linewidth=2)
ax.set_title("Residual Plot for the Best-Fitting Line")
ax.set_xlabel("Predicted stipend ($ thousands)")
ax.set_ylabel("Residual: actual − predicted")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("Mean residual:", round(best_residuals.mean(), 10))


For a useful linear model, residuals should appear as an unstructured cloud around zero. Curvature, a widening spread, or extreme points may signal that a straight-line model is inadequate. Notebook 6.3 will examine regression diagnostics more systematically.


## 15. Guided practice


Complete these tasks:

1. Use the least-squares line to predict the stipend for 7.5 years of experience.
2. Find the residual for the first observation.
3. Calculate the MSE of the least-squares line.
4. Compare the cost of hypotheses `28 + 6x` and `35 + 5x`.
5. Explain why a zero mean residual does not guarantee a good model.


In [ ]:
# Write your solution here.


<details>
<summary><strong>Open the suggested solution</strong></summary>

```python
# 1. Prediction for 7.5 years
print(predict_line(7.5, best_intercept, best_slope))

# 2. Residual for the first observation
print(y[0] - best_predictions[0])

# 3. MSE
print(mean_squared_error(y, best_predictions))

# 4. Compare costs
cost_a = chapter_cost(y, predict_line(X, 28, 6))
cost_b = chapter_cost(y, predict_line(X, 35, 5))
print(cost_a, cost_b)

# 5. Positive and negative errors can balance around zero even when they are large
# or show a systematic pattern.
```

</details>


## 16. Challenge: Find a line by experimentation


Choose your own intercept and slope. Calculate its cost and plot it with the data. Try to bring the cost close to the least-squares minimum without copying the fitted parameters.

Then answer:

1. Which parameter did you adjust first?
2. How did changing the slope affect predictions at large $x$ values?
3. Was visual judgment sufficient to identify the minimum-cost line?


In [ ]:
# Change these two values.
your_intercept = 30
your_slope = 5

your_predictions = predict_line(X, your_intercept, your_slope)
your_cost = chapter_cost(y, your_predictions)

print("Your cost:", round(your_cost, 3))
print("Least-squares cost:", round(chapter_cost(y, best_predictions), 3))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X, y, color="#4C78A8", edgecolor="black", label="Observed data")
ax.plot(X, your_predictions, color="#F58518", linewidth=2.5, label="Your line")
ax.plot(X, best_predictions, color="#E45756", linestyle="--", linewidth=2,
        label="Least-squares line")
ax.set_xlabel("Research experience (years)")
ax.set_ylabel("Annual stipend ($ thousands)")
ax.set_title("Your Candidate Line")
ax.legend()
plt.tight_layout()
plt.show()


## 17. Common mistakes to avoid


| Mistake | Why it is a problem | Better practice |
|---|---|---|
| Treating correlation as causation | Other variables may explain the relationship | Use causal language only with an appropriate design |
| Interpreting slope without units | The meaning becomes unclear | State change in $y$ per one-unit change in $x$ |
| Assuming the intercept is always meaningful | Zero may be outside the realistic range | Interpret it in context |
| Extrapolating far beyond observed data | The linear pattern may not continue | Label and limit extrapolation |
| Looking only at the fitted line | Problems may appear in residuals | Inspect residual patterns |
| Calling every prediction exact | Regression estimates an average relationship | Communicate uncertainty and limitations |


## 18. Reflection


1. What is the difference between $y$ and $\hat{y}$?
2. What do the slope and intercept represent in this example?
3. Why are residuals squared in the cost function?
4. Why does the factor $1/2$ not change the best-fitting line?
5. What is the difference between interpolation and extrapolation?
6. Why can a strong linear relationship still be unsuitable for making compensation decisions?


## 19. Key takeaways


- Simple linear regression models an average relationship between one input and one continuous outcome.
- The hypothesis $\hat{y}=\theta_0+\theta_1x$ defines a straight line.
- The slope measures predicted change in the outcome per unit change in the input.
- The intercept is the predicted outcome at $x=0$, but it may not always be meaningful.
- Residuals are vertical differences between observed and predicted values.
- Least squares selects parameters that minimize squared errors.
- A fitted association does not establish causation, and extrapolation requires caution.

### Looking ahead

In **Notebook 6.2: Linear Regression with Gradient Descent**, you will visualize the cost surface, derive the gradients, implement simultaneous parameter updates, and investigate how the learning rate affects convergence.
